# Telco Customer Churn — Exploratory Data Analysis

This notebook explores the IBM Telco Customer Churn dataset before model training.

**Goals:**
- Understand class balance (churn rate)
- Inspect numerical and categorical distributions
- Identify relationships between features and churn

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import RAW_DATA_PATH

sns.set_theme(style="whitegrid")
df = pd.read_csv(RAW_DATA_PATH)
df.head()

In [ ]:
df.info()
df.describe(include="all").T

In [ ]:
churn_rate = df["Churn"].value_counts(normalize=True)
print(churn_rate)

fig, ax = plt.subplots(figsize=(5, 4))
churn_rate.plot(kind="bar", color=["#2563eb", "#ef4444"], ax=ax)
ax.set_title("Churn Distribution")
ax.set_ylabel("Proportion")
ax.set_xticklabels(["No Churn", "Churn"], rotation=0)
plt.show()

In [ ]:
numeric_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col in zip(axes, numeric_cols):
    sns.histplot(data=df, x=col, hue="Churn", multiple="stack", ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

In [ ]:
categorical_cols = [
    "Contract",
    "PaymentMethod",
    "InternetService",
    "OnlineSecurity",
    "TechSupport",
]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for ax, col in zip(axes, categorical_cols):
    churn_pct = df.groupby(col)["Churn"].value_counts(normalize=True).unstack().fillna(0)
    if "Yes" in churn_pct.columns:
        churn_pct["Yes"].sort_values(ascending=False).plot(kind="bar", ax=ax, color="#ef4444")
    ax.set_title(f"Churn Rate by {col}")
    ax.set_ylabel("Churn Rate")
    ax.tick_params(axis="x", rotation=45)

axes[-1].axis("off")
plt.tight_layout()
plt.show()

## Key Observations

- Churn is imbalanced (~26% churn rate).
- Month-to-month contracts show the highest churn rate.
- Shorter tenure and higher monthly charges correlate with churn.
- Customers without online security or tech support churn more often.